In [ ]:
!pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

In [ ]:
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add t

In [1]:
import vllm

In [2]:
print(vllm.__version__)

0.18.0


In [3]:
!nvidia-smi

Tue Mar 24 15:44:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv,noheader'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('No GPU. Go to Runtime -> Change runtime type -> T4 GPU')
gpu_info = r.stdout.strip()
print('GPU:', gpu_info)
free_mb = int(gpu_info.split(',')[2].strip().replace(' MiB',''))
print('VRAM free:', free_mb, 'MB', '-- OK' if free_mb > 10000 else '-- LOW, restart runtime')


GPU: Tesla T4, 15360 MiB, 14913 MiB
VRAM free: 14913 MB -- OK


In [6]:
import subprocess
def run(cmd, desc):
    print(f'Installing {desc}...', end=' ', flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print('FAILED'); print(r.stderr[-2000:]); raise RuntimeError(desc)
    print('done')
run('pip install -q peft==0.10.0 accelerate', 'PEFT + Accelerate')
run('pip install -q openai httpx rich', 'OpenAI client + rich')
print('All done.')

Installing PEFT + Accelerate... done
Installing OpenAI client + rich... done
All done.


In [7]:
!pip install transformers==4.40.2

# Create LoRA Adpters

In [8]:
import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

BASE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
ADAPTER_CODE_PATH = '/content/adapters/code-assistant'
ADAPTER_SUMM_PATH = '/content/adapters/summarizer'
os.makedirs(ADAPTER_CODE_PATH, exist_ok=True)
os.makedirs(ADAPTER_SUMM_PATH, exist_ok=True)

print('Loading base model')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map='cpu')
print(f'Loaded: {sum(p.numel() for p in base_model.parameters())/1e9:.2f}B params')

# Adapter 1: Code Assistant (rank=8)
print('Creating code-assistant adapter (rank=8)')
m1 = get_peft_model(base_model, LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16, target_modules=['q_proj','v_proj'], lora_dropout=0.05, bias='none'))
tr, tot = m1.get_nb_trainable_parameters()
print(f'  Trainable: {tr/1e6:.2f}M / {tot/1e9:.2f}B ({100*tr/tot:.2f}%)')
m1.save_pretrained(ADAPTER_CODE_PATH); tokenizer.save_pretrained(ADAPTER_CODE_PATH)
del m1

# Adapter 2: Summarizer (rank=16, also adapts k_proj)
print('Creating summarizer adapter (rank=16)')
m2 = get_peft_model(base_model, LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, target_modules=['q_proj','v_proj','k_proj'], lora_dropout=0.05, bias='none'))
tr, tot = m2.get_nb_trainable_parameters()
print(f'  Trainable: {tr/1e6:.2f}M / {tot/1e9:.2f}B ({100*tr/tot:.2f}%)')
m2.save_pretrained(ADAPTER_SUMM_PATH); tokenizer.save_pretrained(ADAPTER_SUMM_PATH)
del m2, base_model

import gc; gc.collect(); torch.cuda.empty_cache()

sz = lambda p: sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(p) for f in fs)/1e6
print(f'Adapter sizes: code={sz(ADAPTER_CODE_PATH):.1f}MB  summarizer={sz(ADAPTER_SUMM_PATH):.1f}MB')
print('Both adapters share the same base model in GPU memory at serve time.')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Loading base model


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded: 1.10B params
Creating code-assistant adapter (rank=8)
  Trainable: 1.13M / 1.10B (0.10%)
Creating summarizer adapter (rank=16)
  Trainable: 3.06M / 1.10B (0.28%)
Adapter sizes: code=4.6MB  summarizer=8.5MB
Both adapters share the same base model in GPU memory at serve time.


In [2]:
!pip install -q --upgrade transformers vllm

In [3]:
import transformers, vllm

print("Transformers:", transformers.__version__)
print("vLLM:", vllm.__version__)

Transformers: 4.57.6
vLLM: 0.18.0


### **Launch vLLM**

In [2]:
import subprocess, time, httpx

BASE_MODEL   = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
ADAPTER_CODE = '/content/adapters/code-assistant'
ADAPTER_SUMM = '/content/adapters/summarizer'
LOG_FILE     = '/content/vllm_server.log'

cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', BASE_MODEL,
    '--port', '8000',
    '--dtype', 'float16',
    '--max-model-len', '1024',
    '--gpu-memory-utilization', '0.65',
    '--enforce-eager',
    '--enable-lora',
    '--max-lora-rank', '16',
    '--lora-modules',
        f'code-assistant={ADAPTER_CODE}',
        f'summarizer={ADAPTER_SUMM}',
]

print('Starting vLLM...')
log_f = open(LOG_FILE, 'w')
server_proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT)

for attempt in range(60):
    time.sleep(2)
    try:
        if httpx.get('http://localhost:8000/health', timeout=3).status_code == 200:
            print(f'Server ready after {(attempt+1)*2}s')
            break
    except Exception:
        pass
    if attempt % 5 == 4:
        print(f'  Still loading... ({(attempt+1)*2}s)')
else:
    with open(LOG_FILE) as f:
        print(''.join(f.readlines()[-30:]))
    raise RuntimeError('vLLM failed to start')

models = httpx.get('http://localhost:8000/v1/models').json()
print('Registered adapters:')
for m in models['data']:
    print(f'  {m["id"]}')


Starting vLLM...
Server ready after 2s
Registered adapters:
  TinyLlama/TinyLlama-1.1B-Chat-v1.0
  code-assistant
  summarizer


Router

In [4]:
from openai import OpenAI
import re, time

client = OpenAI(base_url='http://localhost:8000/v1', api_key='not-needed')

CONFIGS = {
    'code-assistant': {
        'system': 'You are an expert programming assistant. Always respond with clean, well-commented code. Briefly explain what it does first.',
        'temperature': 0.2, 'max_tokens': 400,
    },
    'summarizer': {
        'system': 'You are a concise summarization assistant. Extract key points as a tight structured summary with bullet points.',
        'temperature': 0.4, 'max_tokens': 300,
    },
}

class LoRARouter:
    CODE_RE = re.compile(r'\b(write|code|implement|function|class|script|python|javascript|algorithm|program|debug|fix|refactor|sql|regex)\b', re.I)

    def route(self, msg):
        return 'code-assistant' if self.CODE_RE.search(msg) else 'summarizer'

    def complete(self, msg, adapter=None):
        adapter = adapter or self.route(msg)
        cfg = CONFIGS[adapter]
        t0 = time.perf_counter()
        resp = client.chat.completions.create(
            model=adapter,
            messages=[{'role':'system','content':cfg['system']},{'role':'user','content':msg}],
            temperature=cfg['temperature'], max_tokens=cfg['max_tokens'],
        )
        lat = time.perf_counter() - t0
        toks = resp.usage.completion_tokens
        return {'adapter': adapter, 'content': resp.choices[0].message.content,
                'tokens': toks, 'latency_s': round(lat,2), 'tok_per_s': round(toks/lat,1)}

router = LoRARouter()
print('Router ready. Tests:')
for msg in ['Write a Python function','Summarize this article','implement binary search']:
    print(f'  "{msg}" -> {router.route(msg)}')

Router ready. Tests:
  "Write a Python function" -> code-assistant
  "Summarize this article" -> summarizer
  "implement binary search" -> code-assistant


**Comparison of both Adapters**

In [5]:
from rich.console import Console
from rich.panel import Panel
from rich.columns import Columns
console = Console()

PROMPTS = [
    'Explain how a hash table works.',
    'What is gradient descent?',
    'Describe the difference between TCP and UDP.',
]

console.print('[bold cyan]Multi-LoRA Adapter Comparison[/bold cyan]\n')
for i, prompt in enumerate(PROMPTS, 1):
    console.print(f'[bold yellow]Prompt {i}:[/bold yellow] {prompt}')
    cr = router.complete(prompt, adapter='code-assistant')
    sr = router.complete(prompt, adapter='summarizer')
    console.print(Columns([
        Panel(cr['content'], title=f'[green]code-assistant[/green] {cr["latency_s"]}s', border_style='green', padding=(1,2)),
        Panel(sr['content'], title=f'[blue]summarizer[/blue] {sr["latency_s"]}s', border_style='blue', padding=(1,2)),
    ], equal=True))
    console.print()
console.print('[bold green]Both adapters served from one TinyLlama-1.1B instance.[/bold green]')


Multi-LoRA Adapter Comparison

Prompt 1: Explain how a hash table works.

╭───────────────────────────────────────────── code-assistant 8.52s ──────────────────────────────────────────────╮
│                                                                                                                 │
│  A hash table is a data structure that stores key-value pairs. It is a type of associative array where the      │
│  keys are hashed (hashed) to a fixed number of slots (buckets) in the table. The hash function is used to map   │
│  a key to a bucket index.                                                                                       │
│                                                                                                                 │
│  Here's how a hash table works:                                                                                 │
│                                                                                                                 │
│  1. The key is hashed to a fixed number of slots (buckets) in the table using a hash function.                  │
│                                                                                                                 │
│  2. Each bucket is assigned a unique index (hash value).                                                        │
│                                                                                                                 │
│  3. When a key is inserted into the table, the key is hashed to the correct bucket index.                       │
│                                                                                                                 │
│  4. If the key is already present in the table, the value associated with the key is retrieved from the         │
│  corresponding bucket.                                                                                          │
│                                                                                                                 │
│  5. If the key is not present in the table, the value is inserted into the table using the default value.       │
│                                                                                                                 │
│  In other words, a hash table is a data structure that maps keys to buckets, where each bucket is assigned a    │
│  unique index. When a key is inserted into the table, the key is hashed to the correct bucket index, and if     │
│  the key is already present in the table, the value associated with the key is retrieved from the               │
│  corresponding bucket. If the key is not present in the table, the value is inserted into the table using the   │
│  default value.                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭──────────────────────────────────────────────── summarizer 8.4s ────────────────────────────────────────────────╮
│                                                                                                                 │
│  A hash table is a data structure used to store and retrieve data by using a hash function to map keys to       │
│  unique values. The key is an arbitrary value that is used to identify the corresponding value in the hash      │
│  table.                                                                                                         │
│                                                                                                                 │
│  A hash table works by breaking the input key into a series of bits or bytes, and then using a hash function    │
│  to map those bits or bytes to a unique index in the hash table. The hash function is a mathematical function   │
│  that maps a key to a unique integer value.           

Prompt 2: What is gradient descent?

╭───────────────────────────────────────────── code-assistant 6.65s ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Gradient descent is a popular algorithm used in machine learning and optimization. It is a type of iterative   │
│  algorithm that is used to find the minimum or maximum of a function. Gradient descent works by iteratively     │
│  updating the parameters of a function using a gradient descent step.                                           │
│                                                                                                                 │
│  The algorithm works by taking a step in the direction of the gradient, which is the direction of the largest   │
│  change in the function. Gradient descent updates the parameters of the function by multiplying them by a       │
│  small step size, which is called the learning rate. Gradient descent then updates the parameters using the     │
│  new values of the parameters.                                                                                  │
│                                                                                                                 │
│  Gradient descent is a simple algorithm that can be used to find the minimum or maximum of a function. It is    │
│  often used in machine learning and optimization problems, where the function to be minimized or maximized is   │
│  a function of multiple variables. Gradient descent can be used with both linear and nonlinear functions.       │
│                                                                                                                 │
│  Gradient descent is a popular algorithm in machine learning because it is easy to implement and can be used    │
│  with a wide range of functions. It is also a simple algorithm that can be easily understood by beginners in    │
│  machine learning.                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────── summarizer 3.77s ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Gradient descent is a type of optimization algorithm used in machine learning and deep learning. It is a       │
│  variation of the stochastic gradient descent algorithm, which is a popular optimization method in machine      │
│  learning. Gradient descent is a first-order method, which means that it only updates the parameters of the     │
│  model based on the gradient of the loss function with respect to the parameters. Gradient descent iteratively  │
│  updates the parameters by taking the gradient of the loss function with respect to each parameter, and then    │
│  updating the parameter values using a small step size. Gradient descent is a popular algorithm in machine      │
│  learning because it is efficient, scalable, and can handle large datasets.                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Prompt 3: Describe the difference between TCP and UDP.

╭───────────────────────────────────────────── code-assistant 6.71s ──────────────────────────────────────────────╮
│                                                                                                                 │
│  TCP (Transmission Control Protocol) and UDP (User Datagram Protocol) are two different communication           │
│  protocols used in the internet.                                                                                │
│                                                                                                                 │
│  TCP is a protocol that provides a reliable and ordered delivery of data packets over a network. It uses a      │
│  connection-oriented model, where a single connection is established between two endpoints, and data is         │
│  transmitted in a sequence. TCP provides a mechanism for error handling and congestion control, which helps to  │
│  prevent packet loss and ensure that data is delivered in a timely manner.                                      │
│                                                                                                                 │
│  UDP, on the other hand, is a protocol that provides a unidirectional, datagram-based communication over a      │
│  network. It does not provide a connection-oriented model, and data is transmitted in an unordered manner. UDP  │
│  is designed for small packets of data that do not require a reliable delivery, and it is often used for        │
│  unreliable or low-latency applications.                                                                        │
│                                                                                                                 │
│  In summary, TCP provides a reliable and ordered delivery of data packets over a network, while UDP provides a  │
│  unidirectional, datagram-based communication over a network.                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────── summarizer 6.54s ────────────────────────────────────────────────╮
│                                                                                                                 │
│  TCP (Transmission Control Protocol) and UDP (User Datagram Protocol) are two different types of transport      │
│  protocols used in computer networking. Here's a brief explanation of each:                                     │
│                                                                                                                 │
│  1. TCP: Transmission Control Protocol is a protocol used to manage the flow of data in a network. It ensures   │
│  that data is delivered to its destination in a timely and reliable manner. It is used for sending and          │
│  receiving data over a network.                                                                                 │
│                                                                                                                 │
│  2. UDP: User Datagram Protocol is a protocol used to send and receive data over an unreliable network. It is   │
│  similar to TCP, but it does not guarantee the delivery of data. Instead, it allows data to be sent without     │
│  any guarantees. It is used for sending unstructured data over an unreliable network.                           │
│                                                                                                                 │
│  In summary, TCP ensures the delivery of data to its destination, while UDP does not guarantee the delivery of  │
│  data. TCP is used for sending and receiving data over a network, while UDP is used for sending and receiving   │
│  data over an unreliable network.                     

Both adapters served from one TinyLlama-1.1B instance.

Auto-routing

In [6]:
from rich.console import Console
from rich.panel import Panel
console = Console()

MIXED = [
    'Write a Python function for longest common subsequence.',
    'Summarize the key ideas behind transformer architecture.',
    'Implement a rate limiter using a sliding window.',
    'What are the main takeaways from Attention Is All You Need?',
]

console.print('[bold cyan]Auto-Router Demo[/bold cyan]\n')
for prompt in MIXED:
    r = router.complete(prompt)
    color = 'green' if r['adapter'] == 'code-assistant' else 'blue'
    console.print(Panel(f'[dim]{prompt}[/dim]\n\n{r["content"]}',
        title=f'[{color}]-> {r["adapter"]}[/{color}] {r["latency_s"]}s', border_style=color))
    console.print()

Auto-Router Demo

╭─────────────────────────────────────────── -> code-assistant 11.39s ────────────────────────────────────────────╮
│ Write a Python function for longest common subsequence.                                                         │
│                                                                                                                 │
│ Here's a Python function that finds the longest common subsequence (LCS) between two given strings:             │
│                                                                                                                 │
│ ```python                                                                                                       │
│ def longest_common_subsequence(s1, s2):                                                                         │
│     """                                                                                                         │
│     Finds the longest common subsequence (LCS) between two given strings.                                       │
│     :param s1: first string                                                                                     │
│     :param s2: second string                                                                                    │
│     :return: longest LCS                                                                                        │
│     """                                                                                                         │
│     n = len(s1)                                                                                                 │
│     m = len(s2)                                                                                                 │
│     dp = [[0 for _ in range(m+1)] for _ in range(n+1)]                                                          │
│                                                                                                                 │
│     # Initialize the LCS array                                                                                  │
│     for I in range(n+1):                                                                                        │
│         dp[0] = i                                                                                               │
│                                                                                                                 │
│     for j in range(m+1):                                                                                        │
│         dp[0] = j                                                                                               │
│                                                                                                                 │
│     # Loop through the LCS array                                                                                │
│     for I in range(1, n+1):                                                                                     │
│         for j in range(1, m+1):                                                                                 │
│             if s1 == s2:                                                                                        │
│                 dp = dp + 1                                                                                     │
│             else:                                                                                               │
│                 dp = max(dp, dp)                                                                                │
│                                                                                                                 │
│     return dp                                                                                                   │
│ ```                                                                                                             │
│                                                       

╭────────────────────────────────────────────── -> summarizer 8.51s ──────────────────────────────────────────────╮
│ Summarize the key ideas behind transformer architecture.                                                        │
│                                                                                                                 │
│ Transformer architecture is a type of neural network architecture that employs a transformer module to process  │
│ input tokens and generate output tokens. The transformer module is a self-attention mechanism that allows the   │
│ network to attend to specific parts of the input and generate a response. The transformer architecture is       │
│ designed to be highly efficient and effective in natural language processing (NLP) tasks, such as machine       │
│ translation, question answering, and natural language generation. Some key ideas behind transformer             │
│ architecture include:                                                                                           │
│                                                                                                                 │
│ 1. Self-attention: The transformer module uses a self-attention mechanism to attend to specific parts of the    │
│ input and generate a response. This mechanism allows the network to learn to recognize and understand the       │
│ relationship between different tokens in the input.                                                             │
│                                                                                                                 │
│ 2. Multi-head attention: The transformer module uses multiple attention heads to attend to different parts of   │
│ the input. This allows the network to attend to different aspects of the input and generate a response.         │
│                                                                                                                 │
│ 3. Encoder-decoder architecture: The transformer architecture is based on an encoder-decoder architecture,      │
│ where the encoder generates a sequence of tokens, and the decoder uses the encoder's output to generate a       │
│ response.                                                                                                       │
│                                                                                                                 │
│ 4. Positional encoding: The transformer module uses a positional encoding technique to represent the relative   │
│ position of each token in the input. This allows the network to understand the relationship between different   │
│ tokens in the input.                                                                                            │
│                                                                                                                 │
│ 5. Fine-tuning                                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── -> code-assistant 11.38s ────────────────────────────────────────────╮
│ Implement a rate limiter using a sliding window.                                                                │
│                                                                                                                 │
│ Sure, I'd be happy to help you implement a rate limiter using a sliding window.                                 │
│                                                                                                                 │
│ Here's an example implementation:                                                                               │
│                                                                                                                 │
│ ```python                                                                                                       │
│ import time                                                                                                     │
│                                                                                                                 │
│ def rate_limiter(func, limit, window_size=10):                                                                  │
│     """                                                                                                         │
│     A rate limiter function that limits the number of calls made to a function                                  │
│     per specified window size.                                                                                  │
│                                                                                                                 │
│     :param func: the function to limit                                                                          │
│     :param limit: the maximum number of calls allowed per window size                                           │
│     :param window_size: the window size in seconds                                                              │
│     """                                                                                                         │
│     start_time = time.time()                                                                                    │
│     calls = 0                                                                                                   │
│     while True:                                                                                                 │
│         if time.time() - start_time > window_size:                                                              │
│             calls += 1                                                                                          │
│             if calls >= limit:                                                                                  │
│                 return                                                                                          │
│         func()                                                                                                  │
│                                                                                                                 │
│ def main():                                                                                                     │
│     rate_limiter(lambda: print("Hello, world!"), 10)                                                            │
│                                                                                                                 │
│ if __name__ == "__main__":                                                                                      │
│     main()                                                                                                      │
│ ```                                                                                                             │
│                                                       

╭────────────────────────────────────────────── -> summarizer 6.31s ──────────────────────────────────────────────╮
│ What are the main takeaways from Attention Is All You Need?                                                     │
│                                                                                                                 │
│ Attention Is All You Need by Eli Pariser is a book that explores the ways in which the internet and social      │
│ media are affecting our attention and the consequences of this on our mental health and well-being. Some of the │
│ main takeaways from the book include:                                                                           │
│                                                                                                                 │
│ 1. The rise of short-form content, such as memes and tweets, has led to a decrease in the quality of attention. │
│                                                                                                                 │
│ 2. The constant stream of information and notifications on social media can lead to feelings of overload and    │
│ disconnection.                                                                                                  │
│                                                                                                                 │
│ 3. The impact of social media on our mental health and well-being is significant, with negative effects on      │
│ cognitive function, sleep, and emotional regulation.                                                            │
│                                                                                                                 │
│ 4. There are strategies and tools available to help us manage our attention and improve our mental health, such │
│ as mindfulness practices, time management, and social support.                                                  │
│                                                                                                                 │
│ 5. The book provides a critical analysis of the effects of social media on our democracy and the need for more  │
│ responsible use and regulation of these platforms.                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Benchmark

In [7]:
import time
from rich.console import Console
from rich.table import Table
from rich import box
console = Console()

BENCH = [
    ('Write a Python hello world.', 'code-assistant'),
    ('Summarize: AI is transforming software development.', 'summarizer'),
    ('Implement bubble sort in Python.', 'code-assistant'),
    ('Summarize the concept of neural networks.', 'summarizer'),
    ('Write a string reverse function.', 'code-assistant'),
    ('Summarize: LoRA reduces fine-tuning cost.', 'summarizer'),
]

console.print(f'[bold cyan]Benchmark: {len(BENCH)} requests, alternating adapters[/bold cyan]\n')
results = []
t0 = time.perf_counter()
for prompt, adapter in BENCH:
    r = router.complete(prompt, adapter=adapter)
    results.append(r)
    console.print(f'  [{adapter}] {r["latency_s"]}s / {r["tok_per_s"]} tok/s')

total_time = time.perf_counter() - t0
code_rs = [r for r in results if r['adapter']=='code-assistant']
summ_rs = [r for r in results if r['adapter']=='summarizer']

t = Table(box=box.ROUNDED, title='Benchmark Summary')
t.add_column('Metric', style='cyan'); t.add_column('Value')
t.add_row('Total requests', str(len(results)))
t.add_row('Wall time', f'{total_time:.1f}s')
t.add_row('Total tokens', str(sum(r['tokens'] for r in results)))
t.add_row('Avg latency', f'{sum(r["latency_s"] for r in results)/len(results):.2f}s')
t.add_row('code-assistant avg', f'{sum(r["latency_s"] for r in code_rs)/len(code_rs):.2f}s')
t.add_row('summarizer avg', f'{sum(r["latency_s"] for r in summ_rs)/len(summ_rs):.2f}s')
t.add_row('Adapter switches', str(len(results)-1))
console.print(t)

Benchmark: 6 requests, alternating adapters

4.11s / 35.1 tok/s

2.9s / 35.2 tok/s

10.5s / 35.4 tok/s

7.02s / 35.1 tok/s

4.93s / 34.9 tok/s

1.21s / 33.8 tok/s

      Benchmark Summary       
╭────────────────────┬───────╮
│ Metric             │ Value │
├────────────────────┼───────┤
│ Total requests     │ 6     │
│ Wall time          │ 30.7s │
│ Total tokens       │ 1077  │
│ Avg latency        │ 5.11s │
│ code-assistant avg │ 6.51s │
│ summarizer avg     │ 3.71s │
│ Adapter switches   │ 5     │
╰────────────────────┴───────╯

In [8]:
import signal
server_proc.send_signal(signal.SIGTERM)
server_proc.wait(timeout=10)
import torch, gc; gc.collect(); torch.cuda.empty_cache()
print('Server stopped. GPU memory freed.')

Server stopped. GPU memory freed.
